IMU = ok pour 1 faire boucle
CALIB = ok donne les mat et dist



In [ ]:
import numpy as np
import cv2
import pickle
from traitement_imu import Dataframe_IMU
from ops.generate_cloud import generate_cloud  

In [ ]:

PATH_GAUCHE_CALIB = "..."
PATH_DROITE_CALIB = "..."
CSV_IMU = "..."

In [ ]:
#Ouverture des args pour calibrage
def take_pickle_args(chemin_calibrage: str) -> tuple:
    with open(chemin_calibrage, 'rb') as f:
        l_recupere = pickle.load(f)

    mtx = l_recupere[0]
    dist = l_recupere[1]
    newcameramtx = l_recupere[2]
    roi = l_recupere[3]
    return mtx, dist, newcameramtx, roi

In [ ]:
#convertir angles d'euler en matrice de rotation pour optim de l'IMU
def euler_to_R(roll: float, pitch: float, yaw: float) -> np.ndarray:
    roll_mat = np.array([
        [1, 0, 0],
        [0, np.cos(roll), -np.sin(roll)],
        [0, np.sin(roll),  np.cos(roll)]
    ])

    pitch_mat = np.array([
        [ np.cos(pitch), 0, np.sin(pitch)],
        [0,              1, 0],
        [-np.sin(pitch), 0, np.cos(pitch)]
    ])

    yaw_mat = np.array([
        [np.cos(yaw), -np.sin(yaw), 0],
        [np.sin(yaw),  np.cos(yaw), 0],
        [0,            0,           1]
    ])

    return yaw_mat @ pitch_mat @ roll_mat

In [ ]:
#distorsion
mtx_G, dist_G, newcameramtx_G, roi_G = take_pickle_args(PATH_GAUCHE_CALIB)
mtx_D, dist_D, newcameramtx_D, roi_D = take_pickle_args(PATH_DROITE_CALIB)
K_G = newcameramtx_G
dist_coeffs_G = dist_G

In [ ]:
#form ation du df de l'imu
df_imu_obj = Dataframe_IMU(CSV_IMU)
df_imu_obj.apply()
df_imu = df_imu_obj.get_dataframe().copy()

print(df_imu.head())

In [ ]:
fps = 30
k = 0   

t_image = k / fps * 1000.0   # ms

idx = (df_imu["time_ms"] - t_image).abs().idxmin()

roll = float(df_imu.loc[idx, "roll"])
pitch = float(df_imu.loc[idx, "pitch"])
yaw = float(df_imu.loc[idx, "yaw"])

R_init = euler_to_R(roll, pitch, yaw)
rvec_init, _ = cv2.Rodrigues(R_init)

# translation initiale grossière
tvec_init = np.zeros((3, 1), dtype=np.float64)

In [ ]:
p3d, pts_2d, _ = generate_cloud(PATH_IMG_L, PATH_IMG_R)



print("p3d shape:", p3d.shape)
print("pts_2d shape:", pts_2d.shape)

assert p3d.shape[0] == pts_2d.shape[0]
assert p3d.shape[1] == 3
assert pts_2d.shape[1] == 2

# nettoyage
mask_valid = np.isfinite(p3d).all(axis=1)
p3d = p3d[mask_valid]
pts_2d = pts_2d[mask_valid]

mask_depth = (p3d[:, 2] > 0.2) & (p3d[:, 2] < 10)
p3d = p3d[mask_depth]
pts_2d = pts_2d[mask_depth]

print("Points après clean:", len(p3d))

# normalisation
mean = np.mean(p3d, axis=0)
p3d = p3d - mean

In [ ]:
success, rvec, tvec, inliers = cv2.solvePnPRansac(
    p3d,
    pts_2d,
    K_G,
    dist_coeffs_G,
    rvec=rvec_init,
    tvec=tvec_init,
    useExtrinsicGuess=True,
    iterationsCount=100,
    reprojectionError=3.0,
    confidence=0.99,
    flags=cv2.SOLVEPNP_ITERATIVE
)

if not success or inliers is None:
    raise RuntimeError("PnP failed")

In [ ]:
p3d_in = p3d[inliers[:, 0]]
pts_2d_in = pts_2d[inliers[:, 0]]

success, rvec, tvec = cv2.solvePnP(
    p3d_in,
    pts_2d_in,
    K,
    dist_coeffs,
    rvec=rvec,
    tvec=tvec,
    useExtrinsicGuess=True,
    flags=cv2.SOLVEPNP_ITERATIVE
)

if not success:
    raise RuntimeError("Refined PnP failed")

In [ ]:

proj, _ = cv2.projectPoints(p3d_in, rvec, tvec, K, dist_coeffs)
proj = proj.reshape(-1, 2)

error = np.mean(np.linalg.norm(proj - pts_2d_in, axis=1))
print("Reprojection error:", error)

R_final, _ = cv2.Rodrigues(rvec)
print("R_final =\n", R_final)
print("t_final =\n", tvec)